In [1]:
from dataclasses import dataclass
from enum import Enum
import random
import time
from typing import Any, Callable, Dict, List, Optional, TypedDict


# ==========================================
# 1. 底层 Runtime 契约与异常体系 (Day 28/29)
# ==========================================
class ErrorKind(Enum):
    RETRYABLE = "retryable"
    NON_RETRYABLE = "non_retryable"
    RATE_LIMITED = "rate_limited"
    TIMEOUT = "timeout"
    OVERLOADED = "overloaded"
    CANCELLED = "cancelled"
    INTERNAL = "internal"


@dataclass
class ToolError:
    kind: ErrorKind
    message: str
    retryable: bool
    status_code: Optional[int] = None
    retry_after: Optional[float] = None


@dataclass
class ToolResult:
    ok: bool
    tool_name: str
    data: Any = None
    error: Optional[ToolError] = None
    attempts: int = 0
    latency_ms: float = 0.0


def normalize_error(
    status_code: Optional[int] = None,
    exc: Optional[Exception] = None,
    retry_after: Optional[float] = None,
    is_bulkhead_rejected: bool = False,
    is_circuit_open: bool = False,
) -> ToolError:
    if is_circuit_open:
        return ToolError(
            kind=ErrorKind.OVERLOADED,
            message="CircuitBreaker is OPEN",
            retryable=False,
        )

    if is_bulkhead_rejected:
        return ToolError(
            kind=ErrorKind.OVERLOADED,
            message="Bulkhead capacity full",
            retryable=False,
            status_code=503,
        )

    if exc is not None:
        if isinstance(exc, TimeoutError):
            return ToolError(
                kind=ErrorKind.TIMEOUT,
                message=str(exc) or "Request execution timed out",
                retryable=True,
            )
        return ToolError(
            kind=ErrorKind.INTERNAL,
            message=f"Internal exception: {type(exc).__name__} - {str(exc)}",
            retryable=False,
        )

    if status_code is not None:
        if status_code == 429:
            return ToolError(
                kind=ErrorKind.RATE_LIMITED,
                message="HTTP 429 Too Many Requests",
                retryable=True,
                status_code=429,
                retry_after=retry_after or 0.1,
            )
        if status_code in (500, 502, 503, 504):
            return ToolError(
                kind=ErrorKind.RETRYABLE,
                message=f"HTTP {status_code} Server Error",
                retryable=True,
                status_code=status_code,
            )
        if status_code in (408,):
            return ToolError(
                kind=ErrorKind.TIMEOUT,
                message=f"HTTP {status_code} Request Timeout",
                retryable=True,
                status_code=status_code,
            )
        if status_code in (400, 401, 403, 404, 422):
            return ToolError(
                kind=ErrorKind.NON_RETRYABLE,
                message=f"HTTP {status_code} Client Error",
                retryable=False,
                status_code=status_code,
            )

    return ToolError(
        kind=ErrorKind.INTERNAL,
        message=f"Unknown raw error (status={status_code})",
        retryable=False,
        status_code=status_code,
    )


class PolicyAction(Enum):
    RETRY = "retry"
    FAIL = "fail"
    FALLBACK = "fallback"
    CANCEL = "cancel"
    RATE_LIMIT_WAIT = "rate_limit_wait"
    DEADLINE_EXCEEDED = "deadline_exceeded"


@dataclass
class PolicyDecision:
    action: PolicyAction
    delay: float = 0.0
    reason: str = ""


class PolicyEngine:
    @staticmethod
    def decide(
        error: ToolError,
        retry_count: int,
        max_retries: int,
        base_delay: float,
        remaining_budget: float,
        request_timeout: float,
    ) -> PolicyDecision:
        if not error.retryable:
            return PolicyDecision(action=PolicyAction.FAIL, reason=f"Non-retryable: {error.kind.value}")

        if retry_count >= max_retries:
            return PolicyDecision(action=PolicyAction.FAIL, reason=f"Max retries reached ({max_retries})")

        if error.kind == ErrorKind.RATE_LIMITED:
            computed_delay = error.retry_after if error.retry_after is not None else 0.1
            action = PolicyAction.RATE_LIMIT_WAIT
        else:
            computed_delay = base_delay * (2 ** retry_count) + random.uniform(0.0, 0.01)
            action = PolicyAction.RETRY

        required_budget = computed_delay + request_timeout
        if remaining_budget < required_budget:
            return PolicyDecision(
                action=PolicyAction.DEADLINE_EXCEEDED,
                delay=0.0,
                reason=f"Insufficient remaining budget ({remaining_budget:.3f}s < {required_budget:.3f}s)",
            )

        return PolicyDecision(action=action, delay=computed_delay, reason=f"Allowed {action.value}")


class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown: float = 5.0):
        self.state: str = "CLOSED"
        self.failure_count: int = 0
        self.failure_threshold: int = failure_threshold
        self.cooldown: float = cooldown
        self.opened_at: Optional[float] = None

    def can_call(self) -> bool:
        now = time.time()
        if self.state == "OPEN":
            if self.opened_at and (now - self.opened_at >= self.cooldown):
                self.state = "HALF_OPEN"
                return True
            return False
        return True

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        if self.state == "HALF_OPEN" or self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.opened_at = time.time()


class Bulkhead:
    def __init__(self, capacity: int):
        self.capacity: int = capacity
        self.in_flight: int = 0

    def try_acquire(self) -> bool:
        if self.in_flight < self.capacity:
            self.in_flight += 1
            return True
        return False

    def release(self) -> None:
        if self.in_flight > 0:
            self.in_flight -= 1


@dataclass
class ToolConfig:
    name: str
    max_retries: int
    base_delay: float
    timeout: float
    breaker: CircuitBreaker
    bulkhead: Bulkhead


class ToolRegistry:
    def __init__(self):
        self.configs: Dict[str, ToolConfig] = {}
        self.funcs: Dict[str, Callable] = {}

    def register(self, config: ToolConfig, func: Callable):
        self.configs[config.name] = config
        self.funcs[config.name] = func

    def get_config(self, name: str) -> ToolConfig:
        return self.configs[name]

    def get_func(self, name: str) -> Callable:
        return self.funcs[name]


class ToolRuntime:
    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def execute(self, tool_name: str, args: Dict[str, Any], deadline: Optional[float] = None) -> ToolResult:
        start_time = time.perf_counter()
        config = self.registry.get_config(tool_name)
        tool_fn = self.registry.get_func(tool_name)

        if deadline is None:
            deadline = time.time() + 10.0

        if not config.breaker.can_call():
            elapsed_ms = (time.perf_counter() - start_time) * 1000
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                data=None,
                error=normalize_error(is_circuit_open=True),
                attempts=0,
                latency_ms=elapsed_ms,
            )

        retry_count = 0

        while True:
            if not config.bulkhead.try_acquire():
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=None,
                    error=normalize_error(is_bulkhead_rejected=True),
                    attempts=retry_count,
                    latency_ms=elapsed_ms,
                )

            raw_result = None
            caught_exc = None
            try:
                raw_result = tool_fn(args)
            except Exception as e:
                caught_exc = e
            finally:
                config.bulkhead.release()

            status_code = raw_result.get("status") if isinstance(raw_result, dict) else None
            retry_after = raw_result.get("retry_after") if isinstance(raw_result, dict) else None

            if caught_exc is not None or (status_code and status_code != 200):
                tool_error = normalize_error(status_code=status_code, exc=caught_exc, retry_after=retry_after)
            else:
                tool_error = None

            if tool_error is None:
                config.breaker.record_success()
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=True,
                    tool_name=tool_name,
                    data=raw_result,
                    error=None,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            config.breaker.record_failure()

            remaining_budget = deadline - time.time()
            decision = PolicyEngine.decide(
                error=tool_error,
                retry_count=retry_count,
                max_retries=config.max_retries,
                base_delay=config.base_delay,
                remaining_budget=remaining_budget,
                request_timeout=config.timeout,
            )

            if decision.action == PolicyAction.FAIL:
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=raw_result,
                    error=tool_error,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            if decision.action == PolicyAction.DEADLINE_EXCEEDED:
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                deadline_err = ToolError(
                    kind=ErrorKind.TIMEOUT,
                    message=f"Deadline Exceeded: {decision.reason}",
                    retryable=False,
                )
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=None,
                    error=deadline_err,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            if decision.action in (PolicyAction.RETRY, PolicyAction.RATE_LIMIT_WAIT):
                time.sleep(decision.delay)
                retry_count += 1


# ==========================================
# 2. LangGraph 状态定义 (AgentState)
# ==========================================
class AgentState(TypedDict):
    tool_name: str
    tool_args: dict
    deadline: float
    tool_result: Optional[ToolResult]
    final_answer: Optional[str]
    route: Optional[str]


# ==========================================
# 3. LangGraph Nodes & Conditional Router
# ==========================================
def execute_tool_node(state: AgentState, runtime: ToolRuntime) -> Dict[str, Any]:
    """薄节点：纯粹透传参数并吸纳归一化的 ToolResult"""
    result = runtime.execute(
        tool_name=state["tool_name"],
        args=state["tool_args"],
        deadline=state["deadline"],
    )
    return {"tool_result": result}


def route_tool_result(state: AgentState) -> str:
    """条件路由：仅凭 ToolResult.ok 与 ErrorKind 分流，不处理任何重试/熔断细节"""
    result = state["tool_result"]
    assert result is not None, "tool_result cannot be None"

    if result.ok:
        return "SUCCESS"

    if result.error and result.error.kind in {
        ErrorKind.OVERLOADED,
        ErrorKind.RATE_LIMITED,
        ErrorKind.TIMEOUT,
    }:
        return "FALLBACK"

    return "FAILURE"


def answer_node(state: AgentState) -> Dict[str, Any]:
    data = state["tool_result"].data if state["tool_result"] else None
    return {"final_answer": f"Success Answer: {data}", "route": "SUCCESS"}


def fallback_node(state: AgentState) -> Dict[str, Any]:
    err_msg = state["tool_result"].error.message if state["tool_result"] and state["tool_result"].error else ""
    return {"final_answer": f"Degraded Fallback Answer (Due to {err_msg})", "route": "FALLBACK"}


def error_handler_node(state: AgentState) -> Dict[str, Any]:
    err_msg = state["tool_result"].error.message if state["tool_result"] and state["tool_result"].error else ""
    return {"final_answer": f"Terminal Error Handled: {err_msg}", "route": "FAILURE"}


# 极简 Graph 执行模拟器
def run_agent_graph(state: AgentState, runtime: ToolRuntime) -> AgentState:
    # 1. execute_tool Node
    node_update = execute_tool_node(state, runtime)
    state.update(node_update)

    # 2. Router Condition
    route_decision = route_tool_result(state)

    # 3. Target Branch Node
    if route_decision == "SUCCESS":
        branch_update = answer_node(state)
    elif route_decision == "FALLBACK":
        branch_update = fallback_node(state)
    else:
        branch_update = error_handler_node(state)

    state.update(branch_update)
    return state


# ==========================================
# 4. Mock 工具与 4 个场景集成测试
# ==========================================
class SequenceMockTool:
    def __init__(self, sequence: List[Any]):
        self.sequence = sequence

    def __call__(self, args: Dict[str, Any]) -> Dict[str, Any]:
        item = self.sequence.pop(0) if self.sequence else 200
        if isinstance(item, Exception):
            raise item
        if isinstance(item, dict):
            return item
        return {"status": item, "data": f"Mock Payload for {args}"}


if __name__ == "__main__":
    registry = ToolRegistry()
    runtime = ToolRuntime(registry)

    # 1. Scenario 1: Search 200 -> SUCCESS
    print("=== Scenario 1: Search 200 -> Router SUCCESS ===")
    registry.register(
        ToolConfig(
            name="search_200",
            max_retries=2,
            base_delay=0.01,
            timeout=1.0,
            breaker=CircuitBreaker(),
            bulkhead=Bulkhead(2),
        ),
        SequenceMockTool([200]),
    )
    s1_init: AgentState = {
        "tool_name": "search_200",
        "tool_args": {"q": "langgraph"},
        "deadline": time.time() + 5.0,
        "tool_result": None,
        "final_answer": None,
        "route": None,
    }
    s1_out = run_agent_graph(s1_init, runtime)
    print(f"Graph Route: {s1_out['route']} (Expected: SUCCESS)")
    print(f"Tool Attempts: {s1_out['tool_result'].attempts} (Expected: 1)")
    print(f"Final Answer: {s1_out['final_answer']}\n")

    # 2. Scenario 2: HR 503 -> Runtime 内部 retry -> 200 -> SUCCESS
    print("=== Scenario 2: HR 503 -> Runtime Retry -> 200 -> Router SUCCESS ===")
    registry.register(
        ToolConfig(
            name="hr_503_retry",
            max_retries=2,
            base_delay=0.01,
            timeout=1.0,
            breaker=CircuitBreaker(),
            bulkhead=Bulkhead(2),
        ),
        SequenceMockTool([503, 200]),
    )
    s2_init: AgentState = {
        "tool_name": "hr_503_retry",
        "tool_args": {"action": "query"},
        "deadline": time.time() + 5.0,
        "tool_result": None,
        "final_answer": None,
        "route": None,
    }
    s2_out = run_agent_graph(s2_init, runtime)
    print(f"Graph Route: {s2_out['route']} (Expected: SUCCESS)")
    print(f"Tool Attempts: {s2_out['tool_result'].attempts} (Expected: 2)")
    print(f"Final Answer: {s2_out['final_answer']}\n")

    # 3. Scenario 3: HR Breaker OPEN -> Router FALLBACK
    print("=== Scenario 3: HR Breaker OPEN -> Router FALLBACK ===")
    open_breaker = CircuitBreaker()
    open_breaker.state = "OPEN"
    open_breaker.opened_at = time.time()  # 触发熔断打开

    registry.register(
        ToolConfig(
            name="hr_breaker_open",
            max_retries=2,
            base_delay=0.01,
            timeout=1.0,
            breaker=open_breaker,
            bulkhead=Bulkhead(2),
        ),
        SequenceMockTool([200]),
    )
    s3_init: AgentState = {
        "tool_name": "hr_breaker_open",
        "tool_args": {"action": "write"},
        "deadline": time.time() + 5.0,
        "tool_result": None,
        "final_answer": None,
        "route": None,
    }
    s3_out = run_agent_graph(s3_init, runtime)
    print(f"Graph Route: {s3_out['route']} (Expected: FALLBACK)")
    print(f"Error Kind: {s3_out['tool_result'].error.kind} (Expected: ErrorKind.OVERLOADED)")
    print(f"Final Answer: {s3_out['final_answer']}\n")

    # 4. Scenario 4: HR 400 -> Router FAILURE
    print("=== Scenario 4: HR 400 -> Router FAILURE ===")
    registry.register(
        ToolConfig(
            name="hr_400",
            max_retries=2,
            base_delay=0.01,
            timeout=1.0,
            breaker=CircuitBreaker(),
            bulkhead=Bulkhead(2),
        ),
        SequenceMockTool([400]),
    )
    s4_init: AgentState = {
        "tool_name": "hr_400",
        "tool_args": {"bad_param": True},
        "deadline": time.time() + 5.0,
        "tool_result": None,
        "final_answer": None,
        "route": None,
    }
    s4_out = run_agent_graph(s4_init, runtime)
    print(f"Graph Route: {s4_out['route']} (Expected: FAILURE)")
    print(f"Error Kind: {s4_out['tool_result'].error.kind} (Expected: ErrorKind.NON_RETRYABLE)")
    print(f"Final Answer: {s4_out['final_answer']}")

=== Scenario 1: Search 200 -> Router SUCCESS ===
Graph Route: SUCCESS (Expected: SUCCESS)
Tool Attempts: 1 (Expected: 1)
Final Answer: Success Answer: {'status': 200, 'data': "Mock Payload for {'q': 'langgraph'}"}

=== Scenario 2: HR 503 -> Runtime Retry -> 200 -> Router SUCCESS ===
Graph Route: SUCCESS (Expected: SUCCESS)
Tool Attempts: 2 (Expected: 2)
Final Answer: Success Answer: {'status': 200, 'data': "Mock Payload for {'action': 'query'}"}

=== Scenario 3: HR Breaker OPEN -> Router FALLBACK ===
Graph Route: FALLBACK (Expected: FALLBACK)
Error Kind: ErrorKind.OVERLOADED (Expected: ErrorKind.OVERLOADED)
Final Answer: Degraded Fallback Answer (Due to CircuitBreaker is OPEN)

=== Scenario 4: HR 400 -> Router FAILURE ===
Graph Route: FAILURE (Expected: FAILURE)
Error Kind: ErrorKind.NON_RETRYABLE (Expected: ErrorKind.NON_RETRYABLE)
Final Answer: Terminal Error Handled: HTTP 400 Client Error
